# 08 实践记录：口罩训练数据准备完成

这次完成实际数据准备。原始压缩包保留，训练程序将读取生成的图片和 YOLO 标签。

本课不增加理论，只记住：图片与标签要对应，训练/验证要分开，坐标转换后必须画框检查。

## 1. 实际生成的数据

| 子集 | 图片数 | 正确戴口罩框 | 未戴口罩框 | 佩戴不正确框 |
|---|---:|---:|---:|---:|
| 训练集 | 682 | 2601 | 588 | 98 |
| 验证集 | 171 | 631 | 129 | 25 |

共有 853 张图片、4072 个人工标注目标。目标数大于图片数，因为一张图片可以有多张人脸。

同名图片和标签示例：`train/images/maksssksksss0.png` 与 `train/labels/maksssksksss0.txt`。

三类编号保持：0 with_mask、1 without_mask、2 mask_weared_incorrect。

## 2. 数据处理做了什么

- 按 SHA-256 将完全相同的图片分组，5 组重复图片均不跨训练集与验证集；保留全部原始记录。
- 使用固定种子 42 划分，方便复现。
- 将 12 个超出图片右边界的框裁回边界，保留原始与修正坐标记录。
- 统一按连续图像边界解释 XML 四角，使用中心点和宽高归一化；不额外把中心减 1。
- 图片文件内容不变，原始 archive.zip 不变。

这次处理解决已发现的格式、边界与完全重复问题，不等于全面人工重标，也不能排除所有近似图片和来源相关性。

## 3. 去哪里查看与复查

文件位于原部署项目 `D:/PythonWorkSpace/files/mask_detection/yolov5-face-mask-detection` 中：

| 路径 | 用途 |
|---|---|
| datasets/mask/train | 训练图片和标签 |
| datasets/mask/valid | 验证图片和标签 |
| datasets/mask/manifest.json | 每张图片的归属、哈希、修正与统计记录 |
| datasets/mask/previews/label-check.jpg | 从转换后标签画出的检查图 |
| configs/mask.yaml | 训练程序读取的数据配置 |

在项目根目录、dl-study 环境下，可运行已有检查入口：

```powershell
python scripts/prepare_data.py --check-only
```

已完成检查：文件配对、图片内容、类别编号、归一化范围、坐标还原、逐框与原始 XML 的独立比较、完全重复图片不跨子集，以及数据配置路径。

旧准备脚本已备份，新脚本不会覆盖已有输出目录。

## 4. 下一步：短程试跑

先准备兼容的通用 YOLOv5s 预训练权重，再用少量轮次检查训练、验证、保存和重新加载。

训练前妥善停止网页推理服务，避免争抢内存与显存。正式参数根据实际运行情况调整。

本课结束时的数据准备阶段：尚未下载新的通用权重，也未开始微调。之后完成的首次微调见第九课；网页默认仍使用作者原始口罩权重。

数据准备只能证明输入流程通过检查，不能证明模型已经训练或效果提高。